# Comparación de análisis TMT: Interpolado vs Raw

Este notebook carga dos análisis TMT con diferentes configuraciones de interpolación.

In [1]:
import sys
sys.path.insert(0, '/home/gianluca/Research/datapruebas_analysis')

from pathlib import Path

import pandas as pd

from src import config
from src.visualization.compare_interpolation_analysis import (
    load_interpolated_tmt_trials_by_date,
    load_raw_tmt_trials_by_date,
    compare_metrics,
    INTERPOLATED_DATE,
    RAW_DATE,
)

In [2]:
# Cargar los dos DataFrames SIN filtrar por trials comunes
# Valida que cada análisis tenga la configuración esperada
df_interp = load_interpolated_tmt_trials_by_date(INTERPOLATED_DATE)
df_raw = load_raw_tmt_trials_by_date(RAW_DATE)

print(f"Interpolado ({INTERPOLATED_DATE}):")
print(f"  - Trials: {len(df_interp):,}")
print(f"  - Sujetos: {df_interp['subject_id'].nunique()}")

print(f"\nRaw ({RAW_DATE}):")
print(f"  - Trials: {len(df_raw):,}")
print(f"  - Sujetos: {df_raw['subject_id'].nunique()}")

Excluding 18 subjects without both PART_A and PART_B trials: ['030720fd-dc61-422b-bdf2-2db9457d2f1c', '1d0fc2b7-c2a2-43bd-b00c-ab6b0a7ff722', '1e8869b5-564e-4104-b7e7-84cb6a8edf11', '5f6e5399-a575-4a52-91a7-136356728a8a', '82f8adea-8cec-410e-a7b8-dc3396162402']...
Excluding 18 subjects without both PART_A and PART_B trials: ['030720fd-dc61-422b-bdf2-2db9457d2f1c', '1d0fc2b7-c2a2-43bd-b00c-ab6b0a7ff722', '1e8869b5-564e-4104-b7e7-84cb6a8edf11', '5f6e5399-a575-4a52-91a7-136356728a8a', '82f8adea-8cec-410e-a7b8-dc3396162402']...


Interpolado (2026-01-16_19-20-02):
  - Trials: 5,621
  - Sujetos: 366

Raw (2026-01-16_18-57-15):
  - Trials: 5,607
  - Sujetos: 366


In [3]:
df_interp.head()

,subject_id,trial_id,trial_type,is_valid,trial_order_of_appearance,speed_threshold,dispositivo,mano,dispositivo-config,alcohol-drogas,...,nationality,experiment_origin,device,hand,device_config,alcohol_drugs,treatment,pad_usage,final_comment,start_date
0,df0b8572-3570-4c43-ac9b-017fec2adf22,NEUROPRUEBAS_2,PART_A,True,2,0.198439,Mouse,Derecha,si,no,...,Argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,df0b8572-3570-4c43-ac9b-017fec2adf22,NEUROPRUEBAS_3,PART_B,True,3,0.198439,Mouse,Derecha,si,no,...,Argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,df0b8572-3570-4c43-ac9b-017fec2adf22,NEUROPRUEBAS_6,PART_A,True,6,0.198439,Mouse,Derecha,si,no,...,Argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,df0b8572-3570-4c43-ac9b-017fec2adf22,NEUROPRUEBAS_8,PART_A,True,8,0.198439,Mouse,Derecha,si,no,...,Argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,df0b8572-3570-4c43-ac9b-017fec2adf22,NEUROPRUEBAS_10,PART_A,True,10,0.198439,Mouse,Derecha,si,no,...,Argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df_raw.head()

,subject_id,trial_id,trial_type,is_valid,trial_order_of_appearance,speed_threshold,dispositivo,mano,dispositivo-config,alcohol-drogas,...,nationality,experiment_origin,device,hand,device_config,alcohol_drugs,treatment,pad_usage,final_comment,start_date
0,df0b8572-3570-4c43-ac9b-017fec2adf22,NEUROPRUEBAS_2,PART_A,True,2,0.280745,Mouse,Derecha,si,no,...,Argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,df0b8572-3570-4c43-ac9b-017fec2adf22,NEUROPRUEBAS_3,PART_B,True,3,0.280745,Mouse,Derecha,si,no,...,Argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,df0b8572-3570-4c43-ac9b-017fec2adf22,NEUROPRUEBAS_6,PART_A,True,6,0.280745,Mouse,Derecha,si,no,...,Argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,df0b8572-3570-4c43-ac9b-017fec2adf22,NEUROPRUEBAS_8,PART_A,True,8,0.280745,Mouse,Derecha,si,no,...,Argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,df0b8572-3570-4c43-ac9b-017fec2adf22,NEUROPRUEBAS_10,PART_A,True,10,0.280745,Mouse,Derecha,si,no,...,Argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Análisis de trials que no están en ambos datasets

In [9]:
# Identificar trials en cada dataset
#total trials
total_trials = set(zip(df_interp['subject_id'], df_interp['trial_id'])) | set(zip(df_raw['subject_id'], df_raw['trial_id']))
trials_interp = set(zip(df_interp['subject_id'], df_interp['trial_id']))
trials_raw = set(zip(df_raw['subject_id'], df_raw['trial_id']))

# Trials comunes
common_trials = trials_interp & trials_raw

# Trials solo en interpolado
only_interp = trials_interp - trials_raw

# Trials solo en raw
only_raw = trials_raw - trials_interp

print(f"\nTotal trials: {len(total_trials):,}")
print(f"Trials comunes: {len(common_trials):,}")
print(f"Solo en interpolado: {len(only_interp):,}")
print(f"Solo en raw: {len(only_raw):,}")


Total trials: 5,631
Trials comunes: 5,597
Solo en interpolado: 24
Solo en raw: 10


In [6]:
# Filtrar trials que solo están en un dataset
df_only_interp = df_interp[
    df_interp.apply(lambda r: (r['subject_id'], r['trial_id']) in only_interp, axis=1)
].copy()

df_only_raw = df_raw[
    df_raw.apply(lambda r: (r['subject_id'], r['trial_id']) in only_raw, axis=1)
].copy()

print(f"Trials solo en interpolado: {len(df_only_interp)}")
print(f"Trials solo en raw: {len(df_only_raw)}")

Trials solo en interpolado: 24
Trials solo en raw: 10


In [7]:
df_only_interp

,subject_id,trial_id,trial_type,is_valid,trial_order_of_appearance,speed_threshold,dispositivo,mano,dispositivo-config,alcohol-drogas,...,nationality,experiment_origin,device,hand,device_config,alcohol_drugs,treatment,pad_usage,final_comment,start_date
72,1e200df4-7a6b-4aab-b96b-8e8c8497716a,NEUROPRUEBAS_14,PART_A,True,14,0.302310,Mouse,Derecha,si,no,...,Argentina,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138,9b8b167b-7e64-4347-a579-ad8c2ba4025f,NEUROPRUEBAS_20,PART_A,True,20,0.268376,Mouse,Derecha,si,no,...,Argentina,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
280,b1368f7d-47b5-45dd-83f4-5167eca84158,NEUROPRUEBAS_2,PART_A,True,2,0.111954,padNotebook,Derecha,no,no,...,Argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
876,1aadc486-9800-46c6-93f6-13aa11168a00,NEUROPRUEBAS_12,PART_A,True,12,0.301455,Mouse,Derecha,si,no,...,Ecuatoriana,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
929,b4dc8afd-c6ae-4335-9694-217a9e6f2e05,NEUROPRUEBAS_5,PART_B,True,5,0.395467,Mouse,Derecha,si,si,...,Mexico,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
936,b4dc8afd-c6ae-4335-9694-217a9e6f2e05,NEUROPRUEBAS_12,PART_A,True,12,0.395467,Mouse,Derecha,si,si,...,Mexico,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
967,32d611d1-ad54-4835-99ed-460a443df14c,NEUROPRUEBAS_10,PART_A,True,10,0.203738,Mouse,Derecha,si,no,...,Argentina,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1349,26815196-070f-4f94-a765-5bbecabea52c,NEUROPRUEBAS_10,PART_A,True,10,0.334165,Mouse,Derecha,si,no,...,franco argentino,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1443,6838a4c2-ce67-4bd1-931f-2227776ce2a4,NEUROPRUEBAS_14,PART_A,True,14,0.145882,Mouse,Derecha,si,no,...,Argentina,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1821,b65df3e7-91c6-40b7-a732-49622bfa8684,NEUROPRUEBAS_10,PART_A,True,10,0.198239,padNotebook,Derecha,si,no,...,Colombiano,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_only_raw

,subject_id,trial_id,trial_type,is_valid,trial_order_of_appearance,speed_threshold,dispositivo,mano,dispositivo-config,alcohol-drogas,...,nationality,experiment_origin,device,hand,device_config,alcohol_drugs,treatment,pad_usage,final_comment,start_date
377,leandrogori2000@gmail.com-tmt-plugin_112.csv,NEUROPRUEBAS_2,PART_A,True,2,0.309733,NaN,NaN,NaN,NaN,...,Argentina,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
960,32d611d1-ad54-4835-99ed-460a443df14c,NEUROPRUEBAS_3,PART_B,True,3,0.323079,Mouse,Derecha,si,no,...,Argentina,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3407,48482d6a-a3ca-4c42-bdab-9462f26e6ff8,NEUROPRUEBAS_17,PART_B,True,17,0.338657,Mouse,Derecha,si,no,...,Perú,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3712,4657de88-e8fe-4eca-b171-366b1822d2b3,NEUROPRUEBAS_20,PART_A,True,20,0.480592,Mouse,Izquierda,no,no,...,Argentina,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4826,b17c4fc1-95ed-47b9-8dd5-72ee8705a62c,NEUROPRUEBAS_21,PART_B,True,21,0.297352,Mouse,Derecha,si,no,...,Colombiana,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5046,3cf3fc83-1ee9-4389-9978-fa5845035019,NEUROPRUEBAS_21,PART_B,True,21,0.557009,Mouse,Derecha,si,no,...,peruana,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6304,2697d032-ba2c-45e4-8115-588455f5b491,NEUROPRUEBAS_10,PART_A,True,10,0.328362,Mouse,Derecha,si,no,...,Mexicana,neuropruebas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6612,64bbfefd-5852-4921-8c79-c4f708b9b60a,DATAPRUEBAS_4,PART_A,True,4,0.362631,NaN,NaN,NaN,NaN,...,Argentina,datapruebas,Mouse,Derecha,NaN,NaN,no,NaN,NaN,2024-07-13T16:24:35.058000+00:00
7196,961c65a9-5caa-43c3-8cc4-c4c5484cd1cc,DATAPRUEBAS_8,PART_A,True,8,0.379155,NaN,NaN,NaN,NaN,...,Argentina,datapruebas,padNotebook,Izquierda,NaN,NaN,no,NaN,NaN,2024-02-08T15:36:29.660000+00:00
7459,139efc97-c335-401c-a98f-dd3825ab2d87,DATAPRUEBAS_11,PART_B,True,11,0.277513,NaN,NaN,NaN,NaN,...,Argentina,datapruebas,Mouse,Derecha,NaN,NaN,no,NaN,NaN,2024-02-01T20:24:30.641000+00:00
